In [ ]:
!pip install -U -q google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [ ]:
import json
import re
from google import genai

In [ ]:
API_KEY = input("Enter your Gemini API Key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.6-flash"
print("Gemini client configured successfully!")

In [ ]:
def ask_gemini(prompt):
    interaction = client.interactions.create(
        model=MODEL_NAME,
        input=prompt
    )
    return interaction.output_text

In [ ]:
def clean_json_response(response):
    response = response.strip()
    response = re.sub(
        r"^```json\s*",
        "",
        response,
        flags=re.IGNORECASE
    )
    response = re.sub(
        r"^```\s*",
        "",
        response
    )
    response = re.sub(
        r"\s*```$",
        "",
        response
    )
    return json.loads(response.strip())

In [ ]:
def planner_agent(topic, days):
    prompt = f"""
    You are the Planner Agent in a Multi-Agent Study Assistant.

    Student topic:
    {topic}

    Number of study days:
    {days}

    Create a simple and realistic study plan.

    Requirements:
    1. Divide the topic into important subtopics.
    2. Create a day-by-day study plan.
    3. Include learning, revision, and practice.
    4. Keep it suitable for a college student.
    5. Use clear and simple language.
    """
    return ask_gemini(prompt)

In [ ]:
def teacher_agent(topic):
    prompt = f"""
    You are the Teacher Agent in a Multi-Agent Study Assistant.

    Topic:
    {topic}

    Explain this topic to a college student who is learning it
    for the first time.

    Include:
    1. Simple definition
    2. How it works
    3. Easy real-world example
    4. Important concepts
    5. Important exam points

    Use simple language and clear examples.
    Avoid unnecessary complexity.
    """
    return ask_gemini(prompt)

In [ ]:
def quiz_agent(topic, number_of_questions=5):
    prompt = f"""
    You are the Quiz Agent in a Multi-Agent Study Assistant.

    Topic:
    {topic}

    Generate exactly {number_of_questions} multiple-choice questions.

    Return ONLY valid JSON.
    Do not include Markdown.
    Do not include ```json.
    Do not include any text outside the JSON.

    Use exactly this format:

    {{
        "questions": [
            {{
                "question": "Question text",
                "options": {{
                    "A": "Option A",
                    "B": "Option B",
                    "C": "Option C",
                    "D": "Option D"
                }},
                "answer": "A",
                "explanation": "Short explanation"
            }}
        ]
    }}

    Requirements:
    1. Use four options for every question.
    2. The answer must be A, B, C, or D.
    3. Questions should test important concepts.
    4. Questions should be suitable for a college student.
    """
    response = ask_gemini(prompt)
    return clean_json_response(response)

In [ ]:
def identify_weak_questions(quiz, student_answers):
    weak_questions = []
    for i, question in enumerate(quiz["questions"]):

        correct_answer = (
            question["answer"]
            .strip()
            .upper()
        )
        student_answer = (
            student_answers[i]
            .strip()
            .upper()
        )
        if student_answer != correct_answer:
            weak_questions.append({
                "question": question["question"],
                "your_answer": student_answer,
                "correct_answer": correct_answer,
                "explanation": question["explanation"]
            })
    return weak_questions

In [ ]:
def personalized_review_agent(
    topic,
    plan,
    explanation,
    quiz,
    student_answers,
    score,
    total
):
    percentage = (score / total) * 100
    question_results = []
    for i, question in enumerate(quiz["questions"]):
        correct_answer = (
            question["answer"]
            .strip()
            .upper()
        )
        student_answer = student_answers[i]
        question_results.append({
            "question": question["question"],
            "student_answer": student_answer,
            "correct_answer": correct_answer,
            "is_correct":
                student_answer == correct_answer,
            "explanation":
                question["explanation"]
        })
    prompt = f"""
    You are the Review Agent in a Multi-Agent Study Assistant.

    Topic:
    {topic}

    Study Plan:
    {plan}

    Teacher Explanation:
    {explanation}

    Quiz Results:
    {json.dumps(question_results, indent=2)}

    Student Score:
    {score} out of {total}

    Student Percentage:
    {percentage:.1f}%

    Analyze the student's performance.

    Provide:
    1. Performance summary
    2. Questions answered incorrectly
    3. Concepts that need revision
    4. Important exam points
    5. Personalized revision strategy
    6. Final study advice

    If the score is high, recommend advanced practice.
    If the score is average, recommend targeted revision.
    If the score is low, recommend revisiting the fundamentals.

    Use simple and clear language.
    """
    return ask_gemini(prompt)

In [ ]:
def run_study_assistant(
    topic,
    days,
    number_of_questions=5
):
    print("=" * 60)
    print("📚 MULTI-AGENT STUDY ASSISTANT")
    print("=" * 60)
    print("\n🗓️ Planner Agent is working...")
    plan = planner_agent(topic, days)
    print("\n👨‍🏫 Teacher Agent is working...")
    explanation = teacher_agent(topic)
    print("\n📝 Quiz Agent is working...")
    quiz = quiz_agent(
        topic,
        number_of_questions
    )
    print("\n✅ Study material generated!")
    return plan, explanation, quiz

In [ ]:
topic = input("Enter the topic you want to study: ")
days = int(
    input("How many days do you have to prepare? ")
)
plan, explanation, quiz = run_study_assistant(
    topic,
    days,
    number_of_questions=5
)

In [ ]:
print("\n" + "=" * 60)
print("📝 QUIZ")
print("=" * 60)
for i, question in enumerate(
    quiz["questions"],
    1
):
    print(
        f"\nQuestion {i}: "
        f"{question['question']}"
    )
    for option, text in question["options"].items():
        print(f"{option}. {text}")

In [ ]:
student_answers = []
print("\n===== ANSWER THE QUIZ =====")
for i in range(len(quiz["questions"])):
    while True:
        answer = input(
            f"Question {i + 1} (A/B/C/D): "
        ).strip().upper()
        if answer in ["A", "B", "C", "D"]:
            student_answers.append(answer)
            break
        print("Please enter A, B, C, or D.")

In [ ]:
score = 0
for i, question in enumerate(quiz["questions"]):
    correct_answer = (
        question["answer"]
        .strip()
        .upper()
    )
    if student_answers[i] == correct_answer:
        score += 1
total = len(quiz["questions"])
percentage = (score / total) * 100
print("\n===== QUIZ RESULT =====")
print(f"Score: {score}/{total}")
print(f"Percentage: {percentage:.1f}%")

In [ ]:
weak_questions = identify_weak_questions(
    quiz,
    student_answers
)
print("\n===== WEAK QUESTIONS =====")
if len(weak_questions) == 0:
    print("Excellent! You answered all questions correctly.")
else:
    for i, item in enumerate(
        weak_questions,
        1
    ):
        print(f"\nWeak Question {i}:")
        print("Question:", item["question"])
        print("Your answer:", item["your_answer"])
        print("Correct answer:", item["correct_answer"])
        print("Explanation:", item["explanation"])

In [ ]:
print("\n🔍 Review Agent is analyzing your performance...")
review = personalized_review_agent(
    topic=topic,
    plan=plan,
    explanation=explanation,
    quiz=quiz,
    student_answers=student_answers,
    score=score,
    total=total
)
print("\n===== PERSONALIZED REVIEW =====")
print(review)

In [ ]:
print("\n" + "=" * 70)
print("📚 MULTI-AGENT STUDY ASSISTANT - FINAL REPORT")
print("=" * 70)
print("\n📌 TOPIC:")
print(topic)
print("\n📅 STUDY PLAN:")
print(plan)
print("\n👨‍🏫 TOPIC EXPLANATION:")
print(explanation)
print("\n📝 QUIZ RESULT:")
print(f"Score: {score}/{total}")
print(f"Percentage: {percentage:.1f}%")
print("\n📌 WEAK AREAS:")
if weak_questions:
    for item in weak_questions:
        print("\nQuestion:", item["question"])
        print("Your answer:", item["your_answer"])
        print("Correct answer:", item["correct_answer"])
else:
    print("No weak questions. Excellent work!")
print("\n🔍 PERSONALIZED REVIEW:")
print(review)
print("\n" + "=" * 70)
print("✅ STUDY SESSION COMPLETED")
print("=" * 70)

In [ ]:
report = f"""
========================================
MULTI-AGENT STUDY ASSISTANT
========================================

TOPIC:
{topic}

========================================
STUDY PLAN - PLANNER AGENT
========================================

{plan}

========================================
TOPIC EXPLANATION - TEACHER AGENT
========================================

{explanation}

========================================
QUIZ RESULT
========================================

Score: {score}/{total}
Percentage: {percentage:.1f}%

========================================
WEAK QUESTIONS
========================================
"""
if len(weak_questions) == 0:
    report += """
No incorrect answers.
Excellent performance!
"""
else:
    for i, item in enumerate(weak_questions, 1):
        report += f"""
Question {i}:
{item['question']}

Your Answer:
{item['your_answer']}

Correct Answer:
{item['correct_answer']}

Explanation:
{item['explanation']}

"""

report += f"""
========================================
PERSONALIZED REVIEW - REVIEW AGENT
========================================

{review}

========================================
END OF STUDY SESSION
========================================
"""
with open(
    "study_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)
print("✅ Study report saved as study_report.txt")

In [ ]:
from google.colab import files
files.download("study_report.txt")

In [ ]:
import os
os.makedirs("multi_agent_study_assistant", exist_ok=True)
os.makedirs(
    "multi_agent_study_assistant/reports",
    exist_ok=True
)
print("✅ Project folders created!")

In [ ]:
requirements = """google-genai
streamlit
pyngrok
"""
with open(
    "multi_agent_study_assistant/requirements.txt",
    "w"
) as file:
    file.write(requirements)
print("✅ requirements.txt created!")

In [ ]:
%%writefile app.py
import streamlit as st
import json
import re
from google import genai
# ============================================================
# PAGE SETTINGS
# ============================================================
st.set_page_config(
    page_title="Multi-Agent Study Assistant",
    page_icon="📚",
    layout="wide"
)
st.title("📚 Multi-Agent Study Assistant")
st.write(
    "An AI-powered study assistant that uses four specialized "
    "agents to help students plan, learn, practice, and review."
)
st.divider()
# ============================================================
# GEMINI API KEY
# ============================================================
api_key = st.text_input(
    "🔑 Enter your Gemini API Key",
    type="password"
)
# ============================================================
# RUN APP ONLY AFTER API KEY IS PROVIDED
# ============================================================
if api_key:
    client = genai.Client(api_key=api_key)
    MODEL_NAME = "gemini-3.6-flash"
    # ========================================================
    # GEMINI FUNCTION
    # ========================================================
    def ask_gemini(prompt):
        interaction = client.interactions.create(
            model=MODEL_NAME,
            input=prompt
        )
        return interaction.output_text
    # ========================================================
    # JSON CLEANING
    # ========================================================
    def clean_json_response(response):
        response = response.strip()
        response = re.sub(
            r"^```json\s*",
            "",
            response,
            flags=re.IGNORECASE
        )
        response = re.sub(
            r"^```\s*",
            "",
            response
        )
        response = re.sub(
            r"\s*```$",
            "",
            response
        )
        return json.loads(response.strip())
    # ========================================================
    # AGENT 1 — PLANNER AGENT
    # ========================================================
    def planner_agent(topic, days):
        prompt = f"""
        You are the Planner Agent in a Multi-Agent Study Assistant.

        Student topic:
        {topic}

        Number of study days:
        {days}

        Create a simple and realistic study plan.

        Requirements:
        1. Divide the topic into important subtopics.
        2. Create a day-by-day study plan.
        3. Include learning, revision, and practice.
        4. Keep it suitable for a college student.
        5. Use clear and simple language.
        """
        return ask_gemini(prompt)
    # ========================================================
    # AGENT 2 — TEACHER AGENT
    # ========================================================
    def teacher_agent(topic):
        prompt = f"""
        You are the Teacher Agent in a Multi-Agent Study Assistant.

        Topic:
        {topic}

        Explain this topic to a college student who is learning it
        for the first time.

        Include:
        1. Simple definition
        2. How it works
        3. Easy real-world example
        4. Important concepts
        5. Important exam points

        Use simple language and clear examples.
        Avoid unnecessary complexity.
        """
        return ask_gemini(prompt)
    # ========================================================
    # AGENT 3 — QUIZ AGENT
    # ========================================================
    def quiz_agent(topic, number_of_questions=5):
        prompt = f"""
        You are the Quiz Agent in a Multi-Agent Study Assistant.

        Topic:
        {topic}

        Generate exactly {number_of_questions}
        multiple-choice questions.

        Return ONLY valid JSON.
        Do not include Markdown.
        Do not include ```json.
        Do not include any text outside the JSON.

        Use exactly this format:

        {{
            "questions": [
                {{
                    "question": "Question text",
                    "options": {{
                        "A": "Option A",
                        "B": "Option B",
                        "C": "Option C",
                        "D": "Option D"
                    }},
                    "answer": "A",
                    "explanation": "Short explanation"
                }}
            ]
        }}

        Requirements:
        1. Use four options for every question.
        2. The answer must be A, B, C, or D.
        3. Questions should test important concepts.
        4. Questions should be suitable for a college student.
        """
        response = ask_gemini(prompt)
        return clean_json_response(response)
    # ========================================================
    # FIND WEAK QUESTIONS
    # ========================================================
    def identify_weak_questions(
        quiz,
        student_answers
    ):
        weak_questions = []
        for i, question in enumerate(
            quiz["questions"]
        ):
            correct_answer = (
                question["answer"]
                .strip()
                .upper()
            )
            student_answer = (
                student_answers[i]
                .strip()
                .upper()
            )
            if student_answer != correct_answer:
                weak_questions.append({
                    "question": question["question"],
                    "your_answer": student_answer,
                    "correct_answer": correct_answer,
                    "explanation": question["explanation"]
                })
        return weak_questions
    # ========================================================
    # AGENT 4 — REVIEW AGENT
    # ========================================================
    def personalized_review_agent(
        topic,
        plan,
        explanation,
        quiz,
        student_answers,
        score,
        total
    ):
        percentage = (score / total) * 100
        question_results = []
        for i, question in enumerate(
            quiz["questions"]
        ):
            correct_answer = (
                question["answer"]
                .strip()
                .upper()
            )
            student_answer = student_answers[i]
            question_results.append({
                "question": question["question"],
                "student_answer": student_answer,
                "correct_answer": correct_answer,
                "is_correct":
                    student_answer == correct_answer,
                "explanation":
                    question["explanation"]
            })
        prompt = f"""
        You are the Review Agent in a Multi-Agent Study Assistant.

        Topic:
        {topic}

        Study Plan:
        {plan}

        Teacher Explanation:
        {explanation}

        Quiz Results:
        {json.dumps(question_results, indent=2)}

        Student Score:
        {score} out of {total}

        Student Percentage:
        {percentage:.1f}%

        Analyze the student's performance.

        Provide:
        1. Performance summary
        2. Questions answered incorrectly
        3. Concepts that need revision
        4. Important exam points
        5. Personalized revision strategy
        6. Final study advice

        If the score is high, recommend advanced practice.
        If the score is average, recommend targeted revision.
        If the score is low, recommend revisiting the fundamentals.

        Use simple and clear language.
        """
        return ask_gemini(prompt)
    # ========================================================
    # USER INPUT
    # ========================================================
    st.subheader("🎯 Study Session")
    topic = st.text_input(
        "Enter the topic you want to study"
    )
    days = st.number_input(
        "Number of study days",
        min_value=1,
        max_value=30,
        value=5
    )
    number_of_questions = st.number_input(
        "Number of quiz questions",
        min_value=1,
        max_value=20,
        value=5
    )
    # ========================================================
    # GENERATE STUDY MATERIAL
    # ========================================================
    if st.button("🚀 Generate Study Material"):
        if topic.strip() == "":
            st.warning("Please enter a topic.")
        else:
            with st.spinner(
                "🗓️ Planner Agent is creating your study plan..."
            ):
                plan = planner_agent(
                    topic,
                    days
                )
            with st.spinner(
                "👨‍🏫 Teacher Agent is explaining the topic..."
            ):
                explanation = teacher_agent(
                    topic
                )
            with st.spinner(
                "📝 Quiz Agent is generating questions..."
            ):
                quiz = quiz_agent(
                    topic,
                    number_of_questions
                )
            st.session_state["plan"] = plan
            st.session_state["explanation"] = explanation
            st.session_state["quiz"] = quiz
            st.session_state["topic"] = topic
            st.success(
                "✅ Study material generated successfully!"
            )
    # ========================================================
    # DISPLAY STUDY PLAN
    # ========================================================
    if "plan" in st.session_state:
        st.divider()
        st.subheader("🗓️ Planner Agent")
        st.write(
            st.session_state["plan"]
        )
    # ========================================================
    # DISPLAY EXPLANATION
    # ========================================================
    if "explanation" in st.session_state:
        st.divider()
        st.subheader("👨‍🏫 Teacher Agent")
        st.write(
            st.session_state["explanation"]
        )
    # ========================================================
    # DISPLAY QUIZ
    # ========================================================
    if "quiz" in st.session_state:
        st.divider()
        st.subheader("📝 Quiz Agent")
        quiz = st.session_state["quiz"]
        student_answers = []
        for i, question in enumerate(
            quiz["questions"]
        ):
            st.write(
                f"**Question {i + 1}: "
                f"{question['question']}**"
            )
            options = question["options"]
            answer = st.radio(
                "Choose your answer:",
                list(options.keys()),
                format_func=lambda x:
                    f"{x}. {options[x]}",
                key=f"question_{i}"
            )
            student_answers.append(answer)
        # ====================================================
        # SUBMIT QUIZ
        # ====================================================
        if st.button("📊 Submit Quiz"):
            score = 0
            for i, question in enumerate(
                quiz["questions"]
            ):
                correct_answer = (
                    question["answer"]
                    .strip()
                    .upper()
                )
                if student_answers[i] == correct_answer:
                    score += 1
            total = len(
                quiz["questions"]
            )
            percentage = (
                score / total
            ) * 100
            st.subheader("📊 Quiz Result")
            st.write(
                f"**Score:** {score}/{total}"
            )
            st.write(
                f"**Percentage:** {percentage:.1f}%"
            )
            # ================================================
            # FIND WEAK QUESTIONS
            # ================================================
            weak_questions = identify_weak_questions(
                quiz,
                student_answers
            )
            # ================================================
            # REVIEW AGENT
            # ================================================
            with st.spinner(
                "🔍 Review Agent is analyzing your performance..."
            ):
                review = personalized_review_agent(
                    st.session_state["topic"],
                    st.session_state["plan"],
                    st.session_state["explanation"],
                    quiz,
                    student_answers,
                    score,
                    total
                )
            st.divider()
            st.subheader("🔍 Review Agent")
            st.write(review)
            # ================================================
            # WEAK AREAS
            # ================================================
            st.subheader("📌 Weak Areas")
            if len(weak_questions) == 0:
                st.success(
                    "🎉 Excellent! You answered all questions correctly."
                )
            else:
                for i, item in enumerate(
                    weak_questions,
                    1
                ):
                    with st.expander(
                        f"Question {i}"
                    ):
                        st.write(
                            item["question"]
                        )
                        st.write(
                            f"**Your answer:** "
                            f"{item['your_answer']}"
                        )
                        st.write(
                            f"**Correct answer:** "
                            f"{item['correct_answer']}"
                        )
                        st.write(
                            f"**Explanation:** "
                            f"{item['explanation']}"
                        )

In [ ]:
!ls -lh app.py

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
!streamlit run app.py &>/content/streamlit.log &

In [ ]:
!cat /content/streamlit.log

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3IrEl5kgT0kS4METp1C4L9Pr6yA_5iQBU5cXVW6TDpDqushfE")
print("✅ ngrok authentication configured!")

In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🌐 Open your Streamlit app:")
print(public_url)